# 🏭 Catálogo THC → Base de Datos

Extrae productos del catálogo PDF de [THC Chile](https://thc.cl) usando **OCR con GPU**.

## ¿Qué hace este notebook?
1. 📥 Descarga el catálogo PDF
2. 🧠 OCR página por página con EasyOCR (GPU)
3. 🔍 Detecta productos: código SKU, nombre, precio
4. 📁 Exporta a CSV + Excel (descargables)
5. 🔎 Buscador interactivo de productos
6. 🖼️ Extrae imágenes embebidas (para fase de clasificación)

---
**Instrucciones:** Ejecuta las celdas en orden con `Shift+Enter`

In [ ]:
# ⚙️ Dependencias
!pip install pymupdf pandas openpyxl ipywidgets tqdm -q
# EasyOCR - instalar con pytorch CUDA explícito para evitar conflictos
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
!pip install easyocr -q

In [ ]:
import fitz
import pandas as pd
import requests
from PIL import Image, ImageDraw
import io, os, re, json, warnings
from tqdm.notebook import tqdm
from IPython.display import display, Image as IPImage, HTML
import ipywidgets as widgets
import numpy as np

warnings.filterwarnings('ignore')

try:
    import easyocr
    print(f"✅ easyocr {easyocr.__version__}")
except ImportError:
    print("❌ easyocr no instalado. Revisa la celda anterior.")
    print("   Solucion: ejecuta esta celda y reintenta:")
    print("   !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q")
    print("   !pip install easyocr -q")

print("✅ Imports listos")

---
## 📥 1. Descargar catálogo

In [ ]:
URL = "https://thc.cl/archivos/catalogo-thc-chile-2026.pdf"
PDF_PATH = "catalogo-thc-chile-2026.pdf"

if not os.path.exists(PDF_PATH):
    print("Descargando catálogo...")
    r = requests.get(URL, stream=True)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(PDF_PATH, 'wb') as f:
        for chunk in tqdm(r.iter_content(chunk_size=8192), total=total//8192, unit='KB'):
            f.write(chunk)
    print(f"Descargado: {os.path.getsize(PDF_PATH)/1024/1024:.1f} MB")
else:
    print(f"Ya existe: {os.path.getsize(PDF_PATH)/1024/1024:.1f} MB")

---
## 🔍 2. Inspeccionar PDF

In [ ]:
pdf = fitz.open(PDF_PATH)
print(f"📄 Páginas: {pdf.page_count}")
print(f"📏 Formato: {pdf[0].rect.width:.0f} x {pdf[0].rect.height:.0f} pts")

page = pdf[0]
pix = page.get_pixmap(dpi=100)
display(IPImage(pix.tobytes("png"), width=600))

In [ ]:
sample_text = pdf[0].get_text()
if sample_text.strip():
    print("✅ El PDF tiene texto nativo (no necesita OCR)")
else:
    print("❌ Sin texto nativo → OCR necesario")

---
## 📂 3. Identificar secciones del catálogo

Escanea el PDF para encontrar las secciones/marcas disponibles.

In [ ]:
def identify_sections(pdf):
    """Identifica secciones del catálogo escaneando texto de cada página."""
    sections = {}
    page_previews = []
    
    for i in range(pdf.page_count):
        page = pdf[i]
        text = page.get_text().strip()
        
        if text:
            lines = [l.strip() for l in text.split('\n') if l.strip()]
            preview = ' | '.join(lines[:5])[:120]
        else:
            preview = "[Sin texto nativo]"
        
        page_previews.append({'page': i+1, 'text': preview})
    
    print(f"📄 Páginas escaneadas: {len(page_previews)}\n")
    print(f"{'Pág':>4s}  {'Vista previa del texto'}")
    print("-" * 100)
    for p in page_previews:
        print(f"{p['page']:4d}  {p['text']}")
    
    return page_previews

page_previews = identify_sections(pdf)

---
## 📋 4. Seleccionar sección

Define el rango de páginas de la sección que quieres extraer.

In [ ]:
range_start = widgets.IntSlider(
    value=1, min=1, max=pdf.page_count, step=1,
    description='Desde:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='400px')
)
range_end = widgets.IntSlider(
    value=pdf.page_count, min=1, max=pdf.page_count, step=1,
    description='Hasta:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='400px')
)
output_range = widgets.Output()

def on_range_change(change=None):
    output_range.clear_output()
    with output_range:
        start = range_start.value - 1
        end = range_end.value
        if start >= end:
            print("⚠️ El rango 'Desde' debe ser menor que 'Hasta'")
            return
        n_pages = end - start
        print(f"✅ Seleccionadas {n_pages} páginas (pág {range_start.value} a {range_end.value})")
        print(f"\n📦 Vista previa de la selección:")
        for p in page_previews[start:min(end, start+5)]:
            print(f"  pág {p['page']:3d}: {p['text']}")
        if n_pages > 5:
            print(f"  ... y {n_pages-5} páginas más")

range_start.observe(on_range_change, names='value')
range_end.observe(on_range_change, names='value')

display(widgets.HBox([range_start, range_end]))
display(output_range)
on_range_change()

---
## 🖼️ 5. Extraer imágenes de la sección (300 DPI)

Renderiza cada página de la sección a 300 DPI y extrae las imágenes embebidas.
Los archivos se nombran con el texto detectado en la página.

In [ ]:
SECTION_IMG_DIR = "imagenes_seccion"
os.makedirs(SECTION_IMG_DIR, exist_ok=True)

start_idx = range_start.value - 1
end_idx = range_end.value
total_images = 0
manifest = []

for page_num in tqdm(range(start_idx, end_idx), desc="Extrayendo imágenes sección"):
    page = pdf[page_num]
    
    # Obtener texto de la página para nombrar archivos
    page_text = page.get_text().strip()
    if page_text:
        first_line = [l.strip() for l in page_text.split('\n') if l.strip()][:1]
        safe_name = re.sub(r'[^\w\s-]', '', first_line[0] if first_line else f'pagina_{page_num+1:04d}')
        safe_name = re.sub(r'\s+', '_', safe_name)[:50]
    else:
        safe_name = f"pagina_{page_num+1:04d}"
    
    # Extraer imágenes embebidas
    image_list = page.get_images(full=True)
    for idx, img_info in enumerate(image_list):
        xref = img_info[0]
        base_image = pdf.extract_image(xref)
        img_bytes = base_image["image"]
        ext = base_image["ext"]
        
        fname = f"{safe_name}_img{idx:02d}.{ext}"
        fpath = os.path.join(SECTION_IMG_DIR, fname)
        with open(fpath, "wb") as f:
            f.write(img_bytes)
        
        manifest.append({
            'archivo': fname,
            'pagina': page_num + 1,
            'texto': safe_name,
            'formato': ext,
            'tamaño_kb': round(len(img_bytes) / 1024, 1)
        })
        total_images += 1
    
    # Si no hay imágenes embebidas, renderizar la página completa a 300 DPI
    if not image_list:
        pix = page.get_pixmap(dpi=300)
        fname = f"{safe_name}_render.png"
        fpath = os.path.join(SECTION_IMG_DIR, fname)
        pix.save(fpath)
        
        manifest.append({
            'archivo': fname,
            'pagina': page_num + 1,
            'texto': safe_name,
            'formato': 'png (render 300dpi)',
            'tamaño_kb': round(os.path.getsize(fpath) / 1024, 1)
        })
        total_images += 1

print(f"\n✅ {total_images} imágenes extraídas en {SECTION_IMG_DIR}/")
df_manifest = pd.DataFrame(manifest)
display(df_manifest.head(20))

In [ ]:
if total_images > 0:
    zip_name = f"imagenes_seccion_{range_start.value}-{range_end.value}.zip"
    print(f"📦 Comprimiendo {total_images} imágenes...")
    !zip -r {zip_name} {SECTION_IMG_DIR}/ -q
    
    try:
        from google.colab import files
        files.download(zip_name)
        print(f"✅ Descargando {zip_name}")
    except ImportError:
        print(f"📁 Archivo listo: {zip_name} ({os.path.getsize(zip_name)/1024/1024:.1f} MB)")

---
## 🧠 6. Inicializar EasyOCR (GPU)

In [ ]:
import torch
import easyocr

gpu_available = torch.cuda.is_available()
print(f"🔌 GPU detectada: {gpu_available} ({torch.cuda.get_device_name(0) if gpu_available else 'N/A'})")

print("⏳ Inicializando EasyOCR (español)...")
reader = easyocr.Reader(['es'], gpu=gpu_available)
print("✅ EasyOCR listo")

---
## 🔎 7. Ejecutar OCR página por página

Renderiza cada página a imagen y la procesa con EasyOCR.
Esto toma **5-15 minutos** para ~50-80 páginas.

In [ ]:
def easyocr_to_v2(results):
    """Convierte output de EasyOCR a formato compatible (json-safe)."""
    page = []
    for bbox, text, conf in results:
        if not text.strip():
            continue
        xs = [float(p[0]) for p in bbox]
        ys = [float(p[1]) for p in bbox]
        flat = [min(xs), min(ys), max(xs), max(ys)]
        page.append([flat, [str(text), float(conf)]])
    return [page]


In [ ]:
# ⏩ Si ya ejecutaste el OCR antes, recarga desde los JSON:
# all_text = {}
# for i in range(pdf.page_count):
#     with open(f"ocr_pages/page_{i:04d}.json") as f:
#         all_text[i] = json.load(f)
# print(f"Recargados {len(all_text)} páginas")

---
## 👁️ 8. Explorar resultados OCR

Mueve el slider para ver qué texto detectó el OCR en cada página.

In [ ]:
def show_page(page_num):
    result = all_text.get(page_num)
    if not result or not result[0]:
        display(HTML("<p>⚠️ Sin texto detectado</p>"))
        return
    
    page = pdf[page_num]
    pix = page.get_pixmap(dpi=150)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    draw = ImageDraw.Draw(img)
    
    for line in result[0]:
        bbox = line[0]
        text = line[1][0]
        conf = line[1][1]
        draw.rectangle(bbox, outline="red", width=2)
        draw.text((bbox[0], bbox[1]-12), f"{conf:.0%}", fill="red")
    
    display(img)
    print(f"{'TEXTO':45s} {'CONF':>8s}")
    print("-" * 55)
    for line in sorted(result[0], key=lambda x: (x[0][1], x[0][0])):
        t = line[1][0][:50]
        print(f"{t:45s} {line[1][1]:6.0%}")

widgets.interact(show_page, page_num=widgets.IntSlider(0, 0, pdf.page_count-1, step=1))

---
## 📐 9. Analizar estructura del catálogo

Agrupa los textos detectados por filas (misma coordenada Y) para entender la organización de los productos.

In [ ]:
ROW_TOLERANCE = 25

def get_rows(page_num):
    result = all_text.get(page_num)
    if not result or not result[0]:
        return []
    
    items = []
    for line in result[0]:
        bbox = line[0]
        text = line[1][0].strip()
        if text:
            items.append({
                'x': bbox[0], 'y': bbox[1],
                'x2': bbox[2], 'y2': bbox[3],
                'text': text, 'conf': line[1][1]
            })
    
    items.sort(key=lambda it: (it['y'], it['x']))
    
    rows = []
    if not items:
        return rows
    current_row = [items[0]]
    for it in items[1:]:
        if abs(it['y'] - current_row[-1]['y']) < ROW_TOLERANCE:
            current_row.append(it)
        else:
            rows.append(current_row)
            current_row = [it]
    if current_row:
        rows.append(current_row)
    return rows

def show_structure(page_num):
    rows = get_rows(page_num)
    print(f"📄 Página {page_num+1}: {len(rows)} filas detectadas\n")
    for idx, row in enumerate(rows):
        texts = [f"{it['text']}(x={it['x']:.0f})" for it in sorted(row, key=lambda r: r['x'])]
        print(f"  Fila {idx+1:2d} (y≈{row[0]['y']:.0f}): {' | '.join(texts)}")

widgets.interact(show_structure, page_num=widgets.IntSlider(0, 0, pdf.page_count-1, step=1))

---
## 🔧 10. Parsear productos

Usa patrones para detectar:
- **Código SKU**: letras + números (ej: `THC-123`, `P045`)
- **Precio**: formato chileno (ej: `$12.990`)
- **Nombre**: el texto restante

In [ ]:
# Parámetros ajustables
ROW_TOLERANCE = 25
SKU_RE = re.compile(r"\b\d{4,6}\b")  # SKU: 4-6 dígitos
PRICE_RE = re.compile(r"\b\d{1,3}(?:\.\d{3})+\b")  # precio: 1.295, 10.140

def parse_products(all_text, page_count):
    products = []
    
    for page_num in range(page_count):
        result = all_text.get(page_num)
        if not result or not result[0]:
            continue
        
        items = []
        for line in result[0]:
            bbox = line[0]
            text = line[1][0].strip()
            if not text:
                continue
            items.append({
                'x': bbox[0], 'y': bbox[1],
                'x2': bbox[2], 'y2': bbox[3],
                'text': text, 'conf': line[1][1],
                'page': page_num + 1
            })
        
        items.sort(key=lambda it: (it['y'], it['x']))
        
        rows = []
        if not items:
            continue
        current_row = [items[0]]
        for it in items[1:]:
            if abs(it['y'] - current_row[-1]['y']) < ROW_TOLERANCE:
                current_row.append(it)
            else:
                rows.append(current_row)
                current_row = [it]
        if current_row:
            rows.append(current_row)
        
        for row in rows:
            row.sort(key=lambda it: it['x'])
            
            # Dividir en bloques de producto por gaps grandes en X
            blocks = []
            cur = [row[0]]
            for it in row[1:]:
                if it['x'] - cur[-1]['x'] > 80:  # gap entre columnas
                    blocks.append(cur)
                    cur = [it]
                else:
                    cur.append(it)
            if cur:
                blocks.append(cur)
            
            for block in blocks:
                texts = [it['text'] for it in block]
                full = ' '.join(texts)
                
                # Buscar SKU (primer número de 4-6 dígitos)
                sku = None
                for t in texts:
                    m = SKU_RE.search(t)
                    if m:
                        sku = m.group()
                        break
                
                # Buscar precio (último número con formato miles)
                price = None
                for t in reversed(texts):
                    m = PRICE_RE.search(t)
                    if m:
                        price = m.group()
                        break
                
                # El resto es nombre (sacar SKU y precio del texto)
                name = full
                if sku:
                    name = name.replace(sku, '', 1).strip()
                if price:
                    name = name.replace(price, '', 1).strip()
                name = re.sub(r'\s+', ' ', name).strip()
                
                if sku:
                    products.append({
                        'pagina': page_num + 1,
                        'codigo': sku,
                        'nombre': name,
                        'precio': price or '',
                        'confianza': round(min(it['conf'] for it in block), 2)
                    })
    
    return products

products = parse_products(all_text, pdf.page_count)
df = pd.DataFrame(products)
print(f"✅ {len(df)} productos detectados")
display(df.head(20))

if len(df) > 0:
    print(f"\n📊 Estadísticas:")
    print(f"   Con código: {df['codigo'].str.len().gt(0).sum()}")
    print(f"   Con precio: {df['precio'].str.len().gt(0).sum()}")
    print(f"   Confianza promedio: {df['confianza'].mean():.0%}")
    print(f"\nEjemplos de nombres:")
    for n in df['nombre'].head(20):
        print(f"  - '{n}'")

---
## 📁 11. Exportar a CSV + Excel

In [ ]:
if len(df) > 0:
    df.to_csv("catalogo_thc_productos.csv", index=False, encoding='utf-8-sig')
    print("✅ catalogo_thc_productos.csv")
    
    df.to_excel("catalogo_thc_productos.xlsx", index=False, engine='openpyxl')
    print("✅ catalogo_thc_productos.xlsx")
    
    from google.colab import files
    files.download("catalogo_thc_productos.csv")
else:
    print("⚠️ No hay productos para exportar. Ajusta los parámetros de parseo.")

---
## 🔍 12. Buscador interactivo

In [ ]:
if len(df) > 0:
    @widgets.interact
    def search(termino='', codigo='', mostrar=widgets.Dropdown(options=[10, 20, 50, 100], value=20, description='Mostrar')):
        mask = pd.Series([True] * len(df))
        if termino:
            mask &= df['nombre'].str.contains(termino, case=False, na=False)
        if codigo:
            mask &= df['codigo'].str.contains(codigo, case=False, na=False)
        
        result = df[mask]
        print(f"{len(result)} productos encontrados")
        display(result.head(mostrar))
else:
    print("⚠️ No hay datos. Ejecuta la celda de parseo primero.")

---
## 🖼️ 13. Extraer imágenes embebidas

Extrae las fotos de productos incrustadas en el PDF.
Esto prepara el terreno para clasificarlas con un modelo de IA (fase 2).

In [ ]:
IMG_DIR = "imagenes_extraidas"
os.makedirs(IMG_DIR, exist_ok=True)

total_images = 0
for page_num in tqdm(range(pdf.page_count), desc="Extrayendo imágenes"):
    page = pdf[page_num]
    image_list = page.get_images(full=True)
    
    page_dir = os.path.join(IMG_DIR, f"pagina_{page_num+1:04d}")
    os.makedirs(page_dir, exist_ok=True)
    
    for idx, img_info in enumerate(image_list):
        xref = img_info[0]
        base_image = pdf.extract_image(xref)
        img_bytes = base_image["image"]
        ext = base_image["ext"]
        
        fname = f"img_{idx:03d}.{ext}"
        with open(os.path.join(page_dir, fname), "wb") as f:
            f.write(img_bytes)
        total_images += 1

print(f"✅ {total_images} imágenes extraídas en {IMG_DIR}/")

if total_images > 0:
    print("\n📦 Comprimiendo para descargar...")
    !zip -r imagenes_extraidas.zip {IMG_DIR}/ -q
    from google.colab import files
    files.download("imagenes_extraidas.zip")
    print("✅ Descarga iniciada")

---
## 🎯 Resumen

| Archivo | Contenido |
|---------|----------|
| `catalogo_thc_productos.csv` | Productos detectados (CSV UTF-8) |
| `catalogo_thc_productos.xlsx` | Productos detectados (Excel) |
| `imagenes_extraidas/` | Imágenes por página |
| `ocr_pages/` | OCR crudo por página (backup) |

---
### 🔜 Fase 2: Clasificar imágenes con IA

Cuando estés listo, usamos las imágenes extraídas + un modelo pre-entrenado
(ResNet18) para inferir la categoría del producto (taladro, llave, martillo, etc.)
y enriquecer la base de datos.